In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture

In [3]:
def matriz_conf(y_true, y_pred, labels):
    total_labels = labels
    print(total_labels)
    cm = np.zeros((len(total_labels),len(total_labels)), dtype=int)
    for i in range(len(y_true)):
        cm[y_true[i]][y_pred[i]] += 1

    
    cm = pd.DataFrame(cm, columns=total_labels, index=total_labels)

    cm_transp = pd.DataFrame(np.transpose(cm.to_numpy()), columns=total_labels, index=total_labels)

    for c in cm_transp.columns:
        cm_transp[c] = cm_transp[c]/cm_transp[c].sum()

    cm_porcento = pd.DataFrame(np.transpose(cm_transp.to_numpy()), columns=total_labels, index=total_labels)

    return cm, cm_porcento

def acc(cm, hidden_classes):
    cm_transp = pd.DataFrame(np.transpose(cm.dropna().to_numpy()), columns=cm.columns, index=cm.columns)
    acc = 0
    total = 0
    for c in cm_transp.columns:
        if c not in hidden_classes:
            acc += cm_transp[c][c]
        else:
            acc += cm_transp[c][-1]
        total += cm_transp[c].sum()
    return acc/total

In [4]:
filenames = [0,2,3,4,5]

labels_str = ['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']

filenames

# pd.set_option('future.no_silent_downcasting', True)

[0, 2, 3, 4, 5]

In [5]:
exp_val = []
y_true_all_exp_val = []
for i in range(len(filenames)):
    exp_val.append(pd.read_csv(f'val_{filenames[i]}_GMM_BIC_1_10_scores_corr.csv'))
    y_true_all_exp_val.append(exp_val[i]['Label'].values.tolist())
    exp_val[i] = exp_val[i].drop(columns=['Label'])

In [6]:
from sklearn.metrics import classification_report
ths = [20, 20.5, 21, 21.5, 22, 23, 24, 25]
f1s = [[],[],[],[],[],[],[],[]]
for i in range(len(exp_val)):
    index = 0
    for th in ths:
        y_pred = []
        prog= 0
        for j in range(len(exp_val[i])):
            # print(exp[i][j])
            m = np.nanmax(exp_val[i].values[j])
            # print(m)
            if m > th:
                y_pred.append(exp_val[i].values[j].tolist().index(m))
            else:
                y_pred.append(-1)
        # print(y_pred[:10])

        cm, cm_porcento = matriz_conf(y_true_all_exp_val[i], y_pred, list(range(len(labels_str))) + [-1])
        print(f'th = {th} hidden = {filenames[i]}')
        display(cm)
        tp = cm[-1][filenames[i]]
        fp = cm[-1].sum() - tp
        fn = cm.iloc[filenames[i]].sum() - tp
        tn = cm.drop(columns=[-1]).values.sum() - fn

        acc = (tp+tn)/(tp+fp+tn+fn)
        recall = tp/(tp+fn)
        precision = tp/(tp+fp)
        if precision == 0 or recall == 0:
            f1 = 0
        else:
            f1 = 2*precision*recall/(precision+recall)

        f1s[index].append(f1)
        index += 1
        print(f'th: {th} hidden: {filenames[i]} acc:{acc} recall:{recall} precision:{precision} f1:{f1}')

        true_labels = np.array(y_true_all_exp_val[i])
        true_labels[true_labels == filenames[i]] = -1

        print('MULTICLASS DETECTION')
        print(classification_report(true_labels, y_pred))

[0, 1, 2, 3, 4, 5, -1]
th = 20 hidden = 0


,0,1,2,3,4,5,-1
0,0,59660,0,0,0,0,142568
1,0,125832,45,5,272,72,2310
2,0,3,82044,0,0,0,258
3,0,0,3,60903,0,0,44
4,0,57,0,0,45569,0,164
5,0,9,0,0,10,91,37
-1,0,0,0,0,0,0,0


th: 20 hidden: 0 acc:0.8798494487995138 recall:0.7049864509365666 precision:0.9806508415817747 f1:0.8202779559792756
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.98      0.70      0.82    202228
           1       0.68      0.98      0.80    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       0.99      1.00      0.99     45790
           5       0.56      0.62      0.59       147

    accuracy                           0.88    519956
   macro avg       0.87      0.88      0.87    519956
weighted avg       0.91      0.88      0.88    519956

[0, 1, 2, 3, 4, 5, -1]
th = 20.5 hidden = 0


,0,1,2,3,4,5,-1
0,0,58370,0,0,0,0,143858
1,0,125119,43,5,268,68,3033
2,0,3,82000,0,0,0,302
3,0,0,3,60903,0,0,44
4,0,54,0,0,45569,0,167
5,0,9,0,0,10,87,41
-1,0,0,0,0,0,0,0


th: 20.5 hidden: 0 acc:0.8808418404634238 recall:0.7113653895602983 precision:0.9756722845806911 f1:0.8228144580794056
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.98      0.71      0.82    202228
           1       0.68      0.97      0.80    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       0.99      1.00      0.99     45790
           5       0.56      0.59      0.58       147

    accuracy                           0.88    519956
   macro avg       0.87      0.88      0.87    519956
weighted avg       0.91      0.88      0.88    519956

[0, 1, 2, 3, 4, 5, -1]
th = 21 hidden = 0


...

th: 23 hidden: 5 acc:0.9379197470555201 recall:0.7891156462585034 precision:0.0035842293906810036 f1:0.007136046261265417
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.79      0.01       147
           0       1.00      0.99      0.99    202228
           1       1.00      0.80      0.89    128536
           2       1.00      0.95      0.98     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.99      0.99     45790

    accuracy                           0.94    519956
   macro avg       0.83      0.92      0.81    519956
weighted avg       1.00      0.94      0.96    519956

[0, 1, 2, 3, 4, 5, -1]
th = 24 hidden = 5


,0,1,2,3,4,5,-1
0,198328,0,0,0,0,0,3900
1,404,93820,13,6,157,0,34136
2,0,0,76779,0,0,0,5526
3,0,0,3,60843,0,0,104
4,1,1,0,0,45164,0,624
5,0,5,0,0,11,0,131
-1,0,0,0,0,0,0,0


th: 24 hidden: 5 acc:0.9147889436798499 recall:0.891156462585034 precision:0.0029490556268431597 f1:0.005878657332615329
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.89      0.01       147
           0       1.00      0.98      0.99    202228
           1       1.00      0.73      0.84    128536
           2       1.00      0.93      0.97     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.99      0.99     45790

    accuracy                           0.91    519956
   macro avg       0.83      0.92      0.80    519956
weighted avg       1.00      0.91      0.95    519956

[0, 1, 2, 3, 4, 5, -1]
th = 25 hidden = 5


,0,1,2,3,4,5,-1
0,197952,0,0,0,0,0,4276
1,398,85198,2,5,139,0,42794
2,0,0,74615,0,0,0,7690
3,0,0,3,60431,0,0,516
4,1,0,0,0,44571,0,1218
5,0,5,0,0,11,0,131
-1,0,0,0,0,0,0,0


th: 25 hidden: 5 acc:0.8913177268845826 recall:0.891156462585034 precision:0.002313465783664459 f1:0.004614951032198972
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.89      0.00       147
           0       1.00      0.98      0.99    202228
           1       1.00      0.66      0.80    128536
           2       1.00      0.91      0.95     82305
           3       1.00      0.99      1.00     60950
           4       1.00      0.97      0.98     45790

    accuracy                           0.89    519956
   macro avg       0.83      0.90      0.79    519956
weighted avg       1.00      0.89      0.94    519956



# Média absolute threshold

In [7]:
for i in range(len(f1s)):
    print(f'Média f1 absolute th {ths[i]}: {np.mean(np.array(f1s[i]))}')

Média f1 absolute th 20: 0.6552853198197666
Média f1 absolute th 20.5: 0.6852334280616589
Média f1 absolute th 21: 0.6873741957013512
Média f1 absolute th 21.5: 0.6804262452836611
Média f1 absolute th 22: 0.6679072879012354
Média f1 absolute th 23: 0.6449035573290003
Média f1 absolute th 24: 0.603871844937095
Média f1 absolute th 25: 0.5710755955700382
